# Session X: Lab — Building an Interactive Personal Budget Manager

**Goal:** Combine everything from Sessions I–IX into one complete application.

---

## Exercise 1: Define the Transaction Model

Create a `Transaction` class with:
- `INCOME_CATEGORIES` and `EXPENSE_CATEGORIES` class attributes
- `__init__`: amount, category, description, is_income (default False), auto-set date. Raise `ValueError` if the amount is not a positive, finite number (use `math.isfinite()`) or if the category is not in the list for that transaction type
- `to_dict()` and `@classmethod from_dict()` for serialization
- `__repr__` for readable output
- Type hints on all methods

### How to approach this code

A Transaction class is a blueprint for one record. The two category lists are shared class data, while `__init__` checks the amount and category before storing each object's values; `to_dict()` and `from_dict()` provide the two directions between objects and JSON-friendly dictionaries. `__repr__()` makes objects readable in debugging. The test checks that values survive a serialization round trip and that invalid data is rejected.

In [1]:
import math
from datetime import datetime


class Transaction:
    INCOME_CATEGORIES = ["Salary", "Freelance", "Gift", "Other income"]
    EXPENSE_CATEGORIES = ["Food", "Transport", "Entertainment", "Housing", "Utilities", "Other"]

    def __init__(self, amount: float, category: str, description: str,
                 is_income: bool = False) -> None:
        # TODO: raise ValueError for a bad amount or category, then store the fields
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise ValueError(f"amount must be a number: {amount!r}")
        if not math.isfinite(amount) or amount <= 0:
            raise ValueError(f"amount must be positive, finite: {amount!r}")
        allowed = self.INCOME_CATEGORIES if is_income else self.EXPENSE_CATEGORIES
        if category not in allowed:
            kind = "income" if is_income else "expense"
            raise ValueError(f"{category!r} is not a valid {kind} category; choose from {allowed}")

        self.amount = round(amount, 2)
        self.category = category
        self.description = description
        self.is_income = is_income
        self.date = datetime.now().strftime("%Y-%m-%d")

    def to_dict(self) -> dict:
        # TODO
        return {
            "amount": self.amount,
            "category": self.category,
            "description": self.description,
            "is_income": self.is_income,
            "date": self.date
        }

    @classmethod
    def from_dict(cls, data: dict) -> Transaction:
        # TODO
        t = cls(data["amount"], data["category"], data["description"], is_income = data.get("is_income", False))
        t.date = data.get("date", t.date)
        return t

    def __repr__(self) -> str:
        # TODO
        kind = "income" if self.is_income else "expense"
        return (f"Transaction({self.date}), {kind}, {self.category}, {self.amount:.2f}, {self.description!r}")


# Test
t1 = Transaction(25.50, "Food", "Lunch")
print(t1)
t2 = Transaction(3000, "Salary", "Monthly", is_income=True)
print(t2)
data = t1.to_dict()
t1_copy = Transaction.from_dict(data)
print(f"Round-trip: {t1.amount == t1_copy.amount}")
try:
    Transaction(-5, "Food", "Refund?")
except ValueError as error:
    print("Rejected:", error)
try:
    Transaction(10, "Salary", "Not an expense")
except ValueError as error:
    print("Rejected:", error)

Transaction(2026-11-24), expense, Food, 25.50, 'Lunch'
Transaction(2026-11-24), income, Salary, 3000.00, 'Monthly'
Round-trip: True
Rejected: amount must be positive, finite: -5
Rejected: 'Salary' is not a valid expense category; choose from ['Food', 'Transport', 'Entertainment', 'Housing', 'Utilities', 'Other']


---
## Exercise 2: Input Validation Functions

Write:
- `get_valid_amount(prompt)` — positive, finite number (reject `nan` and `inf` with `math.isfinite()`), catches ValueError
- `get_valid_category(categories)` — numbered menu, validates int input
- `get_valid_description()` — rejects empty strings

### How to approach this code

Validation first converts input text, then checks whether the converted value follows the app's rules. A retry loop lets users recover from nonnumeric amounts; separate checks reject zero, negative, non-finite (`nan`, `inf`) or out-of-range values. Reusable helpers keep those rules consistent across menu options.

In [2]:
import math


def get_valid_amount(prompt: str) -> float:
    # TODO: while True + try/except; reject non-finite and non-positive values
    while True:
        text = input(prompt).strip()
        try:
            amount = float(text)
        except ValueError:
            print(f"'{text}' is not a number.")
        if not math.isfinite(amount):
            print("Please enter a real number")
        elif amount <= 0:
            print("Amount must be greater than 0")
        else:
            return amount

def get_valid_category(categories: list[str]) -> str:
    # TODO: numbered menu + validation
    for number, name in enumerate(categories, start=1):
        print(f"   [{number}] {name}")
    while True:
        text = input("Choose a category number: ").strip()
        try:
            choice = int(text)
        except ValueError:
            print(f"'{text}' is not a whole number. Try again.")
            continue
        if 1 <= choice <= len(categories):
            return categories[choice - 1]
        print(f"Please chooe a number from 1 to {len(categories)}.")


def get_valid_description() -> str:
    # TODO: reject empty
    while True:
        text = input("Description: ").strip()
        if text: 
            return text
        print("The description can't be empty.")


# Uncomment to test:
print(get_valid_amount("Amount: $"))

5.0


---
## Exercise 3: JSON Persistence

Write `save_transactions(transactions, filepath)` and `load_transactions(filepath)`.
- Save through a temporary file and `os.replace()`, so a failed write never destroys the old file.
- A missing file (`FileNotFoundError`) is a normal first run: return `[]`.
- A file that is not valid JSON (`JSONDecodeError`), or valid JSON that is not a list, is damaged: move it to `filepath + ".bak"`, print a warning and return `[]`.

### How to approach this code

The save and load functions make in-memory transactions persist between program runs. Writing to a temporary file and swapping it in with `os.replace()` keeps the old file intact if the write fails. `FileNotFoundError` means a first run, while a damaged file is set aside as `.bak` so it is never overwritten. The test uses temporary paths to avoid changing real data.

In [3]:
import json
import os


def save_transactions(transactions: list[dict], filepath: str = "data/budget.json") -> None:
    # TODO: write a temporary file, then os.replace() it into place
    folder = os.path.dirname(filepath)
    if folder:
        os.makedirs(folder, exist_ok = True)
    temp_path = filepath + ".tmp"
    try:
        with open(temp_path, "w", encoding="utf-8") as f:
            json.dump(transactions, f, indent=2)
        os.replace(temp_path, filepath)
    except OSError:
        if os.path.exists(temp_path):
            os.remove(temp_path)
        raise


def load_transactions(filepath: str = "data/budget.json") -> list[dict]:
    # TODO: missing file -> []; damaged or wrong shape -> move to .bak, warn, []
    try:
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
    except FileNotFoundError:
        return []
    except json.JSONDecodeError:
        data = None

    if isinstance(data, list):
        return data

    backup = filepath + ".bak"
    os.replace(filepath, backup)
    print(f"Warning: {filepath} was damaged. It has been moved to {backup}; starting with no records.")
    return []

# Test
test_data = [{"amount": 50, "category": "Food", "description": "Groceries", "is_income": False, "date": "2026-10-01"}]
save_transactions(test_data, "/tmp/test_budget.json")
loaded = load_transactions("/tmp/test_budget.json")
print(loaded)
print(load_transactions("/tmp/nonexistent.json"))
with open("/tmp/damaged_budget.json", "w", encoding="utf-8") as f:
    f.write('{"amount": 5,')
print(load_transactions("/tmp/damaged_budget.json"))
print(os.path.exists("/tmp/damaged_budget.json.bak"))

[{'amount': 50, 'category': 'Food', 'description': 'Groceries', 'is_income': False, 'date': '2026-10-01'}]
[]
[]
True


---
## Exercise 4: CSV Export

Write `export_to_csv(transactions, filepath)` using csv.DictWriter. Create the folder if it is missing, open the file with `newline=""` and `encoding="utf-8"`, and handle an empty list.

### How to approach this code

CSV export maps each transaction dictionary to a row with a stable set of columns. `writeheader()` labels the fields and `writerows()` writes records; the empty-list check prevents a false success message. Reading the temporary export back verifies the actual output.

In [4]:
import csv

COLUMN_NAMES = ["date", "category", "description", "amount", "is_income"]

def export_to_csv(transactions: list[dict], filepath: str = "data/export.csv") -> None:
    # TODO
    if not transactions:
        print("Nothing to export - No transactions")
        return
    folder = os.path.dirname(filepath)
    if folder:
        os.makedirs(folder, exist_ok=True)
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMN_NAMES)
        writer.writeheader()
        writer.writerows(transactions)
    print(f"Exported {len(transactions)} transaction(s) to {filepath}")


# Test
test_data = [{"date":"2026-10-01","category":"Food","description":"Groceries","amount":50.0,"is_income":False}]
export_to_csv(test_data, "/tmp/test_export.csv")
with open("/tmp/test_export.csv", encoding="utf-8") as f:
    print(f.read())
export_to_csv([])

Exported 1 transaction(s) to /tmp/test_export.csv
date,category,description,amount,is_income
2026-10-01,Food,Groceries,50.0,False

Nothing to export - No transactions


---
## Exercise 5: Summary Report

Write `generate_report(transactions)` that calculates income/expenses/balance, groups expenses by category and shows percentages. Handle an empty list, and print no "Expenses by Category" heading for an income-only list.

### How to approach this code

The report calculates income, expenses, balance, and totals by category from the same transaction list. Generator expressions select records before summing; a dictionary accumulates category totals. Print the category block only when there is at least one expense: then `expenses > 0`, so the percentage division is safe, and an income-only report has no empty heading.

In [7]:
def generate_report(transactions: list[dict]) -> None:
    # TODO
    if not transactions:
        print("No transactions to report.")
        return

    income = sum(t["amount"] for t in transactions if t["is_income"])
    expenses = sum(t["amount"] for t in transactions if not t["is_income"])
    balance = income - expenses

    print()
    print("BUDGET REPORT")
    print("--------------------------------------")
    print(f"Income:   {income:>12,.2f}")
    print(f"Expenses: {expenses:>12,.2f}")
    print(f"Balance:  {balance:>12,.2f}")

    if expenses > 0:
        by_category = {}
        for t in transactions:
            if not t["is_income"]:
                by_category[t["category"]] = by_category.get(t["category"], 0) + t["amount"]
        print("\nExpenses by Category:")
        for category, total in sorted(by_category.items(), key=lambda kv: kv[1], reverse=True):
            print(f"  {category:<14}{total:>10,.2f}  ({total / expenses:.1%})")


test_data = [
    {"amount": 3000, "category": "Salary", "description": "Monthly", "is_income": True},
    {"amount": 800, "category": "Housing", "description": "Rent", "is_income": False},
    {"amount": 150, "category": "Food", "description": "Groceries", "is_income": False},
    {"amount": 75, "category": "Food", "description": "Restaurants", "is_income": False},
    {"amount": 50, "category": "Transport", "description": "Metro", "is_income": False},
]
generate_report(test_data)
generate_report([])
generate_report(test_data[:1])   # income only: no "Expenses by Category" heading


BUDGET REPORT
--------------------------------------
Income:       3,000.00
Expenses:     1,075.00
Balance:      1,925.00

Expenses by Category:
  Housing           800.00  (74.4%)
  Food              225.00  (20.9%)
  Transport          50.00  (4.7%)
No transactions to report.

BUDGET REPORT
--------------------------------------
Income:       3,000.00
Expenses:         0.00
Balance:      3,000.00


---
## Exercise 6: Main Application Loop

Wire everything together with a menu: Add Income, Add Expense, View All, Report, Export CSV, Save & Quit, Quit.
- At start-up, load the saved records and convert them with `clean_records()`, which skips (and reports) any record that fails validation.
- `ask_transaction(is_income)` asks for the amount, a category from the right list, and a description.
- Report, export and save work with dictionaries: convert with `[t.to_dict() for t in transactions]`.
- Q asks before discarding unsaved changes; any other key prints an error message.

### How to approach this code

The main loop coordinates the application: it displays choices and calls the relevant validation, model, reporting, export, and storage operations. Keep those implementations in their own functions so each menu branch remains easy to read. Saving before exit differs from quitting without saving, so Q should warn when there are unsaved changes.

In [9]:
def clean_records(records: list[dict]) -> list[Transaction]:
    # TODO: from_dict() each record; report and skip the ones that raise an error
    clean = []
    for number, record in enumerate(records, start=1):
        try:
            clean.append(Transaction.from_dict(record))
        except (KeyError, TypeError, ValueError) as error:
            print(f"Skipping saved record {number}: {error}")
    return clean


def ask_transaction(is_income: bool) -> Transaction:
    # TODO: amount, a category from the right list, description
    amount = get_valid_amount("Amount: ")
    categories = Transaction.INCOME_CATEGORIES if is_income else Transaction.EXPENSE_CATEGORIES
    category = get_valid_category(categories)
    description = get_valid_description()
    return Transaction(amount, category, description, is_income=is_income)


def main() -> None:
    transactions = clean_records(load_transactions())  # TODO: load the saved records and clean them
    unsaved = False
    while True:
        print("\n=== PERSONAL BUDGET MANAGER ===")
        print("[1] Add Income  [2] Add Expense  [3] View All")
        print("[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit")
        choice = input("Select: ").strip().lower()
        if choice == "1":
            # TODO
            transactions.append(ask_transaction(is_income=True))
            unsaved = True
            print("Income added.")
        elif choice == "2":
            # TODO
            transactions.append(ask_transaction(is_income=False))
            unsaved = True
            print("Expense added.")
        elif choice == "3":
            # TODO
            if not transactions:
                print("No transactions yet.")
            for number, t in enumerate(transactions, start=1):
                print(f"{number:>3}. {t!r}")
        elif choice == "4":
            # TODO
            generate_report([t.to_dict() for t in transactions])
        elif choice == "5":
            # TODO
            export_to_csv([t.to_dict() for t in transactions])
        elif choice == "6":
            # TODO: save and break
            save_transactions([t.to_dict() for t in transactions])
            print(f"Saved {len(transactions)} transactions. Goodbye!")
            break
        elif choice == "q":
            # TODO: if there are unsaved changes, ask before quitting
            if unsaved:
                answer = input("You have unsaved changes. Quit without saving? (y/n): ").strip().lower()
                if answer != "y":
                    continue
            print("Goodbye!")
            break
        else:
            print("Invalid option.")

# Uncomment to run (Ctrl-D or the stop button ends the input):
main()


=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
   [1] Salary
   [2] Freelance
   [3] Gift
   [4] Other income
Income added.

=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
   [1] Food
   [2] Transport
   [3] Entertainment
   [4] Housing
   [5] Utilities
   [6] Other
Expense added.

=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit
  1. Transaction(2026-11-24), income, Salary, 100.00, 'Money'
  2. Transaction(2026-11-24), expense, Transport, 30.00, 'Ubers'

=== PERSONAL BUDGET MANAGER ===
[1] Add Income  [2] Add Expense  [3] View All
[4] Report  [5] Export CSV  [6] Save & Quit  [Q] Quit

BUDGET REPORT
--------------------------------------
Income:         100.00
Expenses:        30.00
Balance:         70.00

Expenses by Category:
  Transp